# MoE SFT — Mixture of Experts Fine-tuning

Bu notebook **smoke test ile kanıtlanmış** çalışır (`smoke/07_moe_smoke.py`).

## Dürüst Uyarı: 16GB VRAM Sınırı

Production MoE modelleri 16GB'da **eğitilemez** — Unsloth bnb-4bit'i MoE expert parametrelerinde desteklemiyor:

| Model | Total/Active | 16GB? | Sebep |
|---|---|---|---|
| Qwen3-30B-A3B | 30B/3.3B | ❌ | bnb-4bit yok → 17.5GB minimum (bf16) |
| Qwen3.5-35B-A3B | 35B/3B | ❌ | 74GB |
| Gemma-4-26B-A4B-it | 26B/4B | ❌ | bnb-4bit yok |
| Qwen3-16B-A3B | 16B/3B | ❌ | Hub'da sadece GGUF |
| **`imdatta0/tiny_qwen3_moe_2.8B_0.7B`** | **2.8B/0.7B** | ✅ | T4-uyumlu teaching model |

## Bu Notebook
- **Pipeline öğretiyor** — gerçek MoE patternleri
- TinyQwen3 MoE ile (Unsloth'un kendi `TinyQwen3_MoE.ipynb`'inde kullandığı T4-uyumlu model)
- Production'da Qwen3-30B-A3B için **A100/H100 gerekiyor** — kod aynı, sadece model adı değişir

## Smoke Test Sonuçları (RTX 4070 Ti SUPER 16GB)
- Peak VRAM: **9.64 GB** (16GB rahat)
- Pipeline: TRL 0.24 ile çalıştı (resmi notebook'lar `trl==0.22.2` pin'liyor — bizim stack uyumlu)
- 3 step / Train loss: 12.1 (model weights teaching-grade, gerçek training değil)

## İçindekiler
1. **MoE Nedir?** — Routing + Sparse Experts
2. **Setup + `UNSLOTH_MOE_DISABLE_AUTOTUNE`**
3. **Model — TinyQwen3 MoE (2.8B/0.7B)**
4. **MoE-Specific LoRA** — `gate_up_proj` target_modules'a
5. **Dataset — OpenMathReasoning-mini**
6. **SFTTrainer — Standart**
7. **Training**
8. **Production'a Geçiş — A100 ile**
9. **Yaygın Tuzaklar**

## 1. MoE Nedir?

**Mixture of Experts** — model'in her layer'ında **N expert** var, **router** her token için en iyi K tane seçer.

### Yapı
```
[Input Token] → Router → [Expert 1, 2, ..., N]
                  ↓
        En iyi K expert seçildi (genelde top-2)
                  ↓
        Sadece bu K expert hesap yapar
                  ↓
             [Output Token]
```

### Avantajlar
- **Parametre çok ama hesap az** — Qwen3-30B-A3B = 30B parametre var ama her token sadece **3.3B** kullanıyor
- **Daha hızlı inference** — küçük dense model gibi compute, büyük model gibi knowledge

### Dezavantajlar
- **VRAM = total params**, hesap = active params → bellek tasarrufu YOK
- **Routing overhead** — small batch'te ek maliyet
- **Expert imbalance** — bazı expertlar az kullanılır, dengelemek zor

### Unsloth'un MoE Optimizasyonu (Faster MoE 2026)
- **Split LoRA reordering** — `Y = X @ loraA; Z = Y @ loraB` (full delta materialize değil)
- **3 backend:** `grouped_mm` (default), `unsloth_triton`, `native_torch`
- **2x hız + 35% VRAM** vs vanilla transformers MoE
- **12x hız** vs older transformers v4

## 2. Setup

**Kritik env var:** `UNSLOTH_MOE_DISABLE_AUTOTUNE=1` — autotune memory yiyor, küçük GPU'larda kapatılmalı.

In [ ]:
import os
os.environ['UNSLOTH_MOE_DISABLE_AUTOTUNE'] = '1'    # MUTLAKA unsloth import'undan ONCE

import unsloth
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

print(f'torch: {torch.__version__} | cuda: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

## 3. Model — TinyQwen3 MoE

**KRİTİK 2 ayar:**
- `load_in_4bit=False` — bnb-4bit MoE expert parametrelerinde **desteklenmiyor** (HARD KURAL)
- `fast_inference=False` — vLLM MoE'de **henüz yok**

16GB için tek seçenek bu küçük model. Production için Qwen3-30B-A3B'yi sadece model adını değiştirerek kullan (A100+ gerekir).

In [ ]:
MODEL = 'imdatta0/tiny_qwen3_moe_2.8B_0.7B'         # T4-uyumlu
max_seq_length = 2048
lora_rank = 32

model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL,
    max_seq_length = max_seq_length,
    load_in_4bit = False,                            # MoE: 4-bit YOK
    fast_inference = False,                          # MoE: vLLM YOK
)
print(f'Model: {sum(p.numel() for p in model.parameters())/1e9:.2f}B params')

## 4. MoE-Specific LoRA

Standart SFT'den **2 fark var:**

1. **`target_modules` listesine `gate_up_proj` eklendi** — MoE expert MLP'leri
2. **`lora_alpha = r * 2`** — resmi notebook recipe (r=32 → alpha=64)

**Router NOT trained by default** — Unsloth router fine-tuning'i kapalı tutuyor (`use_router=False` implicit). Router fine-tune etmek expert imbalance riski getirir, genelde önerilmez.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = [
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
        'gate_up_proj',                               # MoE expert layers — KRITIK
    ],
    lora_alpha = lora_rank * 2,                       # 2x — resmi recipe
    use_gradient_checkpointing = True,
    random_state = 3407,
    bias = 'none',
)

In [ ]:
tokenizer = get_chat_template(tokenizer, chat_template='qwen-3')

## 5. Dataset — OpenMathReasoning-mini

Tüm resmi MoE notebook'ları aynı dataset'i kullanıyor. Math reasoning + DeepSeek-R1 trace'leri (`<think>...`).

In [ ]:
dataset = load_dataset('unsloth/OpenMathReasoning-mini', split='cot')
print(f'Total rows: {len(dataset)}')

def to_messages(example):
    return {'conversations': [
        {'role': 'user', 'content': example['problem']},
        {'role': 'assistant', 'content': example['generated_solution']},
    ]}
dataset = dataset.map(to_messages, remove_columns=dataset.column_names)

def fmt(examples):
    return {'text': [
        tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
        for c in examples['conversations']
    ]}
dataset = dataset.map(fmt, batched=True)
print(f'\nFirst 200 chars:')
print(dataset[0]['text'][:200])

## 6. SFTTrainer — Standart

MoE'a özel SFTConfig ayarı **yok** — normal TRL config.

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = 'text',
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 1,
        warmup_steps = 5,
        max_steps = 50,                              # demo; production: num_train_epochs=1
        learning_rate = 2e-4,
        logging_steps = 5,
        optim = 'adamw_8bit',
        weight_decay = 0.001,
        lr_scheduler_type = 'linear',
        seed = 3407,
        report_to = 'none',
    ),
)

## 7. Training

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_mem = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_mem = round(gpu_stats.total_memory / 1024**3, 3)
print(f'GPU = {gpu_stats.name} | start mem = {start_mem} / {max_mem} GB')

trainer_stats = trainer.train()

used = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
print(f"\n=== RESULTS ===")
print(f"Train runtime: {trainer_stats.metrics['train_runtime']:.2f} sec")
print(f"Train loss:    {trainer_stats.metrics['train_loss']:.4f}")
print(f"Peak VRAM:     {used} GB")

## 8. Production'a Geçiş — A100 ile

**Kod tamamen aynı**, sadece model adı değişir:

```python
# Production (A100 80GB)
MODEL = 'unsloth/Qwen3-30B-A3B-Instruct-2507'
max_seq_length = 2048
lora_rank = 32

model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL,
    max_seq_length = max_seq_length,
    load_in_4bit = False,                            # AYNI — MoE 4-bit yok
    fast_inference = False,
)
# get_peft_model + SFTTrainer hepsi aynı
```

### Production Donanım Tahmini (Resmi Unsloth)
| Model | bf16 LoRA VRAM |
|---|---|
| Qwen3-30B-A3B | 17.5 GB (en iyi durum) |
| Qwen3-30B-A3B-Instruct-2507 | ~63 GB |
| Qwen3.5-35B-A3B | 74 GB |
| Qwen3-235B-A22B | 256+ GB |
| Qwen3.5-122B-A10B | 256 GB |

**A100 40GB:** Qwen3-30B-A3B base sığabilir.  
**A100 80GB:** Qwen3-30B-A3B-Instruct-2507 rahat.  
**H100/H200:** Qwen3.5-35B-A3B.

## 9. Yaygın Tuzaklar

| # | Hata | Sonuç | Çözüm |
|---|---|---|---|
| 1 | `load_in_4bit=True` MoE'de | NotImplementedError | `False` zorunlu |
| 2 | `fast_inference=True` | Crash | `False` zorunlu |
| 3 | `target_modules`'ta `gate_up_proj` yok | Expert MLP train edilmez | Listede olmalı |
| 4 | `UNSLOTH_MOE_DISABLE_AUTOTUNE=0` (default) | Autotune memory peak'leri yapar | `=1` set et |
| 5 | `Qwen3.5-7B-A1B` modelini kullanmak | Hub'da YOK | Smallest = Qwen3-30B-A3B |
| 6 | `Qwen3-16B-A3B` safetensors | Sadece GGUF, fine-tune edilmez | A100+ ile 30B-A3B |
| 7 | Router fine-tune etmek | Expert imbalance, model collapse | Default kapalı bırak |
| 8 | Resmi notebook'lar `trl==0.22.2` pin | Eski API kullanır | Bizim stack `trl 0.24` ile çalıştı (smoke kanıtı) |
| 9 | `lora_alpha = r` (standart SFT) | Suboptimal speed | MoE için `r*2` |
| 10 | Multi-GPU MoE training | Henüz desteklenmiyor | Single-GPU only |

## Özet

1. **MoE = Sparse experts + router** — büyük model, az hesap, ama **VRAM = total params**
2. **bnb-4bit MoE'de YOK** → 16GB sınırlı
3. **TinyQwen3 MoE = T4-uyumlu teaching model** — gerçek pattern, küçük model
4. **`gate_up_proj` target_modules'a ekle** — MoE expert MLPs
5. **`lora_alpha = r * 2`** — resmi MoE recipe
6. **`UNSLOTH_MOE_DISABLE_AUTOTUNE=1`** consumer GPU için zorunlu
7. **Production = A100+** — kod aynı, model adı değişir
8. **Router default kapalı** — fine-tune etmemek daha güvenli

**Test edildi:** `smoke/07_moe_smoke.py` — TRL 0.24 ile pipeline çalıştı, 9.64 GB VRAM (RTX 4070 Ti SUPER 16GB).

## Bir sonraki adım: Production MoE Training
1. A100/H100 kirala (vast.ai, Lambda, RunPod, etc.)
2. `MODEL = 'unsloth/Qwen3-30B-A3B-Instruct-2507'` değiştir
3. `max_steps = 50` → `num_train_epochs = 1`
4. Aynı kod, gerçek model